# POMDP agent simplified model — integrated v3 clean

Runs CUDA `traincuda`, original CPU `train(use_gpu=False)`, original CuPy `train(use_gpu=True)`, then policy evaluation histories for all three.

The package applies the non-layered `hist.plot()` fix internally; no notebook metadata patch cells are required.

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd

PATCH_ROOT = os.environ.get(
    "P_TRAINCUDA_ROOT",
    "/home/jlpfritas/HPC-POMDP/v1train_cuda/package_P_traincuda_integrated_v3_1_clean",
)
PATCH_PY = os.path.join(PATCH_ROOT, "python")
if PATCH_PY not in sys.path:
    sys.path.insert(0, PATCH_PY)

# Avoid old cached v1/v2 modules if the kernel was reused.
for name in list(sys.modules):
    if name.startswith("olfnav_cuda_notebook") or name.startswith("olfnav_cuda_backend"):
        del sys.modules[name]

from olfactory_navigation import Environment
from olfactory_navigation.agents import FSVI_Agent
from olfactory_navigation.agents.model_based_util.environment_converter import minimal_converter

import olfnav_cuda_notebook as ocn
from olfnav_cuda_notebook import (
    enable_cuda_backend,
    clean_start_points,
    run_policy_evaluation,
    run_policy_full_evaluation,
    show_cuda_training_report,
)

CUDA_LIB = os.path.join(PATCH_ROOT, "build", "libpomdp_backup_cuda.so")
print("Using patch module from:", ocn.__file__)
print("CUDA_LIB:", CUDA_LIB)
print("CUDA_LIB exists:", os.path.exists(CUDA_LIB))
assert "integrated_v3_clean" in ocn.__file__ or "package_P_traincuda_integrated_v3_1_clean" in ocn.__file__

## Configuration

In [ ]:
ENV_PATH = (
    "/home/jlpfritas/HPC-POMDP/v3/recon/"
    "Env-olfnav-trim552-fliplr-cropx780-thr1e4/"
    "Env-320_485-marg_20_20_20_20-edge_wrap_vertical-start_odor_present-source_163_406_radius2"
)

PARTITIONS_TEST = (24, 24)
EXPANSIONS = 100
GAMMA = 0.95
SEED = 123
CUDA_DEVICE = 0
CUDA_VERSION = "auto"

RUN_CUDA_TRAIN = True
RUN_CPU_TRAIN = True
RUN_CUPY_TRAIN = True
RUN_FULL_EVAL = False
N_EVAL = 100
HORIZON = 1000

OUT_ROOT = Path("tmp") / f"integrated_v3_clean_p{PARTITIONS_TEST[0]}x{PARTITIONS_TEST[1]}_e{EXPANSIONS}"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
OUT_ROOT

## Agent factory

In [ ]:
def make_agent(partitions=PARTITIONS_TEST, seed=SEED, gamma=GAMMA, env_path=ENV_PATH, margin_partitions=True):
    print(f"[MAKE_AGENT] env_path={env_path}")
    print(f"[MAKE_AGENT] partitions={partitions}")
    print(f"[MAKE_AGENT] seed={seed}")
    print(f"[MAKE_AGENT] gamma={gamma}")

    env = Environment.load(env_path)
    ocn.normalize_non_layered_environment(env)  # package-level safe metadata normalization

    ag = FSVI_Agent(
        env,
        environment_converter=minimal_converter,
        partitions=list(partitions),
        margin_partitions=margin_partitions,
        seed=seed,
    )
    ag.gamma = gamma
    try:
        ag.model.gamma = gamma
    except Exception:
        pass
    return ag

def describe_agent(agent, name="agent"):
    env = agent.environment
    print(f"[{name}] env shape:", getattr(env, "shape", None))
    print(f"[{name}] env dimensions:", getattr(env, "dimensions", None))
    print(f"[{name}] layer labels:", getattr(env, "environment_layer_labels", getattr(env, "layer_labels", None)))
    print(f"[{name}] source position:", getattr(env, "source_position", None))
    print(f"[{name}] start_probabilities shape:", np.asarray(env.start_probabilities).shape)

ag_preview = make_agent()
describe_agent(ag_preview, "preview")
try:
    ag_preview.environment.plot()
except Exception as e:
    print("Environment plot skipped:", type(e).__name__, e)

## Common training kwargs

In [ ]:
COMMON_TRAIN_KWARGS = dict(
    expansions=EXPANSIONS,
    max_belief_growth=10,
    prune_level=1,
    prune_interval=10,
    gamma=GAMMA,
    print_progress=True,
    print_stats=True,
    history_tracking_level=1,
)
COMMON_TRAIN_KWARGS

## CUDA traincuda

In [ ]:
ag_cuda = None
res_cuda = None

if RUN_CUDA_TRAIN:
    if not os.path.exists(CUDA_LIB):
        raise FileNotFoundError(f"Missing CUDA_LIB={CUDA_LIB}. Build with: bash scripts/31_build_backend_lib.sh --arch native --clean")
    ag_cuda_base = make_agent()
    ag_cuda = enable_cuda_backend(
        ag_cuda_base,
        device=CUDA_DEVICE,
        version=CUDA_VERSION,
        gamma=GAMMA,
        lib_path=CUDA_LIB,
    )
    print("traincuda available:", hasattr(ag_cuda, "traincuda"))
    print("CUDA config:", ag_cuda._cuda_backend_config)
    res_cuda = ag_cuda.traincuda(
        **COMMON_TRAIN_KWARGS,
        use_gpu=True,
        outdir=str(OUT_ROOT / "cuda_traincuda"),
        checkpoint_every=25,
        visual=True,
        display_rows=10,
    )
else:
    print("CUDA training skipped")

## Original CPU/native train

In [ ]:
ag_cpu = None
hist_cpu_train = None
if RUN_CPU_TRAIN:
    ag_cpu = make_agent()
    hist_cpu_train = ag_cpu.train(**COMMON_TRAIN_KWARGS, use_gpu=False, overwrite_training=True)
    print(hist_cpu_train.summary)
else:
    print("CPU training skipped")

## Original CuPy/native GPU train

In [ ]:
ag_cupy = None
hist_cupy_train = None
if RUN_CUPY_TRAIN:
    import cupy as cp
    cp.cuda.Device(CUDA_DEVICE).use()
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()
    ag_cupy = make_agent()
    hist_cupy_train = ag_cupy.train(**COMMON_TRAIN_KWARGS, use_gpu=True, overwrite_training=True)
    print(hist_cupy_train.summary)
else:
    print("CuPy training skipped")

## Training comparison

In [ ]:
def safe_len_value_function(agent):
    if agent is None:
        return None
    vf = getattr(agent, "value_function", None)
    if vf is None and hasattr(agent, "native_agent"):
        vf = getattr(agent.native_agent, "value_function", None)
    if vf is None:
        return None
    for fn in (lambda x: len(x), lambda x: len(x.alpha_vectors), lambda x: x.alpha_vector_array.shape[0]):
        try:
            return int(fn(vf))
        except Exception:
            pass
    return None

def safe_len_belief(agent):
    if agent is None:
        return None
    obj = agent.native_agent if hasattr(agent, "native_agent") else agent
    for attr in ("belief_set", "belief", "beliefs"):
        b = getattr(obj, attr, None)
        if b is None:
            continue
        for fn in (lambda x: len(x), lambda x: x.belief_array.shape[0], lambda x: x.array.shape[0]):
            try:
                return int(fn(b))
            except Exception:
                pass
    return None

rows = []
for label, ag, res in [("cuda_traincuda", ag_cuda, res_cuda), ("cpu_native", ag_cpu, None), ("cupy_native_gpu", ag_cupy, None)]:
    if ag is None:
        continue
    rows.append({
        "path": label,
        "partitions": f"{PARTITIONS_TEST[0]}x{PARTITIONS_TEST[1]}",
        "expansions": EXPANSIONS,
        "alpha_final": safe_len_value_function(ag),
        "belief_final": safe_len_belief(ag),
        "wall_s": None if res is None else res.summary.get("total_wall_s"),
        "sum_backup_ms": None if res is None else res.summary.get("sum_backup_ms"),
    })

df_train_compare = pd.DataFrame(rows)
display(df_train_compare)
df_train_compare.to_csv(OUT_ROOT / "training_compare_summary.csv", index=False)
if res_cuda is not None:
    res_cuda.to_dataframe().to_csv(OUT_ROOT / "cuda_traincuda_rows.csv", index=False)

## Shared clean starts

In [ ]:
ref_agent = ag_cuda or ag_cpu or ag_cupy
start_points_clean = clean_start_points(ref_agent)
start_points_eval = start_points_clean if N_EVAL is None else start_points_clean[:int(N_EVAL)]
print("clean start points:", start_points_clean.shape)
print("eval start points:", start_points_eval.shape)
start_points_eval[:5]

## Policy evaluation histories

In [ ]:
COMMON_EVAL_KWARGS = dict(
    horizon=HORIZON,
    reward_discount=GAMMA,
    use_gpu=False,
    time_shift=False,
    time_loop=False,
    print_progress=True,
    print_stats=True,
)

hist_cuda_eval = run_policy_evaluation(ag_cuda, start_points=start_points_eval, n=len(start_points_eval), **COMMON_EVAL_KWARGS) if ag_cuda is not None else None
hist_cpu_eval = run_policy_evaluation(ag_cpu, start_points=start_points_eval, n=len(start_points_eval), **COMMON_EVAL_KWARGS) if ag_cpu is not None else None
hist_cupy_eval = run_policy_evaluation(ag_cupy, start_points=start_points_eval, n=len(start_points_eval), **COMMON_EVAL_KWARGS) if ag_cupy is not None else None

## Native hist.plot outputs

In [ ]:
if hist_cuda_eval is not None:
    print("CUDA policy evaluation plot")
    hist_cuda_eval.plot()

if hist_cpu_eval is not None:
    print("CPU/native policy evaluation plot")
    hist_cpu_eval.plot()

if hist_cupy_eval is not None:
    print("CuPy/native GPU policy evaluation plot")
    hist_cupy_eval.plot()

## Optional full evaluation

In [ ]:
hist_cuda_full = hist_cpu_full = hist_cupy_full = None
if RUN_FULL_EVAL:
    hist_cuda_full = run_policy_full_evaluation(ag_cuda, **COMMON_EVAL_KWARGS) if ag_cuda is not None else None
    hist_cpu_full = run_policy_full_evaluation(ag_cpu, **COMMON_EVAL_KWARGS) if ag_cpu is not None else None
    hist_cupy_full = run_policy_full_evaluation(ag_cupy, **COMMON_EVAL_KWARGS) if ag_cupy is not None else None
else:
    print("RUN_FULL_EVAL=False")

In [ ]:
if hist_cuda_full is not None:
    hist_cuda_full.plot()
if hist_cpu_full is not None:
    hist_cpu_full.plot()
if hist_cupy_full is not None:
    hist_cupy_full.plot()

## Save metadata

In [ ]:
metadata = {
    "patch_root": PATCH_ROOT,
    "cuda_lib": CUDA_LIB,
    "env_path": ENV_PATH,
    "partitions": list(PARTITIONS_TEST),
    "expansions": EXPANSIONS,
    "gamma": GAMMA,
    "n_eval": None if N_EVAL is None else int(N_EVAL),
    "horizon": HORIZON,
    "module_file": ocn.__file__,
}
(OUT_ROOT / "notebook_run_metadata.json").write_text(json.dumps(metadata, indent=2, sort_keys=True) + "
")
print("Saved:", OUT_ROOT)